In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.3):
        super().__init__()

        self.fc1 = nn.Linear(in_features, out_features)
        self.bn1 = nn.BatchNorm1d(out_features)

        self.fc2 = nn.Linear(out_features, out_features)
        self.bn2 = nn.BatchNorm1d(out_features)

        self.dropout = nn.Dropout(dropout)

        # Projection if dimensions differ
        self.shortcut = nn.Sequential()
        if in_features != out_features:
            self.shortcut = nn.Sequential(
                nn.Linear(in_features, out_features),
                nn.BatchNorm1d(out_features)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.fc1(x)
        out = self.bn1(out)
        out = F.relu(out)
        out = self.dropout(out)

        out = self.fc2(out)
        out = self.bn2(out)

        out += identity
        out = F.relu(out)

        return out

In [ ]:
class WideResNetTabular(nn.Module):
    def __init__(self, input_dim, num_classes, width=256, depth=4, dropout=0.3):
        super().__init__()

        self.input_layer = nn.Sequential(
            nn.Linear(input_dim, width),
            nn.BatchNorm1d(width),
            nn.ReLU()
        )

        # Stack residual blocks
        self.blocks = nn.Sequential(
            *[ResidualBlock(width, width, dropout) for _ in range(depth)]
        )

        self.head = nn.Sequential(
            nn.Linear(width, width // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(width // 2, num_classes)
        )

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [ ]:
dft = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_t.csv')
dfn = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_n.csv')
dfm = pd.read_csv('/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/encoded_train_m.csv')
#dfv = pd.read_csv("/content/drive/MyDrive/CaresAI/tnm_staging/data_v2/bert_encoding/validation_enc.csv")

In [ ]:
dfm.head(3)

,patient_filename,text,text_embedding_clinicalbert,text_embedding_biobert,text_embedding_pubmedbert,m
0,TCGA-05-4244.3a844132-f813-4d8e-8f7d-dbae0b23d7fd,Diagnosis: 1. Resection material from the righ...,"[0.1355716437101364, -0.12100697308778763, -0....","[0.04545216262340546, -0.03954649344086647, -0...","[-0.06522239744663239, 0.07896429300308228, -0...",1.0
1,TCGA-05-4249.7e920317-d5c2-4160-9e2b-ef0101eb5a23,Diagnosis: 2. and 3. Two-part resection materi...,"[0.1757894605398178, -0.17660431563854218, -0....","[0.1323431432247162, -0.16728118062019348, -0....","[-0.11425544321537018, 0.08171500265598297, -0...",0.0
2,TCGA-05-4250.5574f2f8-f247-40e6-a285-7793edcf5358,2. and 3. Resection material from the right lo...,"[0.19550728797912598, -0.07438599318265915, -0...","[0.12437687069177628, -0.16940933465957642, 0....","[-0.08555292338132858, 0.09673506021499634, -0...",0.0


In [ ]:
ym = dfm['m']
Xm = dfm[['text_embedding_clinicalbert', 'text_embedding_biobert', 'text_embedding_pubmedbert']]
X_train, X_test, y_train, y_test = train_test_split(Xm, ym, test_size=0.20, stratify=ym)

In [ ]:
print(y_train.value_counts(), y_test.value_counts())

m
0.0    2919
1.0     213
Name: count, dtype: int64 m
0.0    731
1.0     53
Name: count, dtype: int64


In [ ]:
import ast

def parse_list_string(list_string):
    return ast.literal_eval(list_string)

X_train_list = []

for index, row in X_train.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_train_list.append(combined_vector)

X_train_m = np.array(X_train_list)
y_train_m = y_train.values

print(f"Shape of X_train: {X_train_m.shape}")
print(f"Shape of y_train: {y_train_m.shape}")

Shape of X_train: (3132, 2304)
Shape of y_train: (3132,)


In [ ]:
X_test_list_m = []
# text_embedding_clinicalbert, text_embedding_biobert, text_embedding_pubmedbert
for index, row in X_test.iterrows():
    combined_vector = np.concatenate([
        parse_list_string(row['text_embedding_clinicalbert']),
        parse_list_string(row['text_embedding_biobert']),
        parse_list_string(row['text_embedding_pubmedbert'])
    ])
    X_test_list_m.append(combined_vector)

X_test_m = np.array(X_test_list_m)
y_test_m = y_test.values

print(f"Shape of X_train: {X_test_m.shape}")
print(f"Shape of y_train: {y_test_m.shape}")

Shape of X_train: (784, 2304)
Shape of y_train: (784,)


In [ ]:
#y_train_t = y_train_t - y_train_t.min()
#y_test_t = y_test_t - y_test_t.min()
print(type(y_train_m[0]), y_train_m[:10])

<class 'numpy.float64'> [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]


In [ ]:
train_dataset = TabularDataset(X_train_m, y_train_m)
val_dataset = TabularDataset(X_test_m, y_test_m)

In [ ]:
num_classes = len(np.unique(y_train))
print("Unique classes:", np.unique(y_train))
print("Num classes:", num_classes)

Unique classes: [0. 1.]
Num classes: 2


In [ ]:
print("Unique labels:", set(y_train_m))
print("Num classes:", num_classes)
print("Max label:", max(y_train_m))

Unique labels: {np.float64(0.0), np.float64(1.0)}
Num classes: 2
Max label: 1.0


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [ ]:
# width=512, 768
model = WideResNetTabular(
    input_dim=X_train_m.shape[1],
    num_classes=num_classes,
    width=256,
    depth=4,
    dropout=0.3
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

WideResNetTabular(
  (input_layer): Sequential(
    (0): Linear(in_features=2304, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (blocks): Sequential(
    (0): ResidualBlock(
      (fc1): Linear(in_features=256, out_features=256, bias=True)
      (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=256, out_features=256, bias=True)
      (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (shortcut): Sequential()
    )
    (1): ResidualBlock(
      (fc1): Linear(in_features=256, out_features=256, bias=True)
      (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (fc2): Linear(in_features=256, out_features=256, bias=True)
      (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_run

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
criterion = torch.nn.CrossEntropyLoss()

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode  # "max" for metrics like F1/AUC
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def step(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improvement = (score - self.best_score) if self.mode == "max" else (self.best_score - score)

        if improvement > self.min_delta:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1

        if self.counter >= self.patience:
            self.early_stop = True

        return self.early_stop

In [ ]:
def evaluate_metrics(model, loader, num_classes):
    model.eval()

    all_preds = []
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device) # Add this line to move input to device
            logits = model(X_batch)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # F1 scores
    f1_macro = f1_score(all_labels, all_preds, average="macro")
    f1_micro = f1_score(all_labels, all_preds, average="micro")
    f1_weighted = f1_score(all_labels, all_preds, average="weighted")

    # AUC (multiclass)
    # For binary classification, roc_auc_score expects probabilities of the positive class (class 1)
    auc = roc_auc_score(all_labels, all_probs[:, 1]) # Corrected: pass only probabilities for class 1
    print(np.isnan(all_probs).any())

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "f1_weighted": f1_weighted,
        "auc": auc
    }

In [ ]:
def train_model(model, train_loader, val_loader, optimizer, criterion, num_classes, epochs=50):
    early_stopper = EarlyStopping(patience=7, mode="max")

    best_f1 = 0
    best_metrics = None
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        metrics = evaluate_metrics(model, val_loader, num_classes)

        print(f"Epoch {epoch+1}")
        print(f"Loss: {avg_loss:.4f}")
        print(f"F1 Macro: {metrics['f1_macro']:.4f}")
        print(f"F1 Weighted: {metrics['f1_weighted']:.4f}")
        print(f"AUC: {metrics['auc']:.4f}")
        print("-" * 40)

        # Track best model (based on F1 macro)
        if metrics["f1_macro"] > best_f1:
            best_f1 = metrics["f1_macro"]
            best_metrics = metrics
            best_epoch = epoch + 1

            # Optional: save best model
            torch.save(model.state_dict(), "best_model.pt")

        # Early stopping
        if early_stopper.step(metrics["f1_macro"]):
            print("Early stopping triggered.")
            break

    # Print best results at the end
    print("\n===== BEST MODEL =====")
    print(f"Best Epoch: {best_epoch}")
    print(f"Best F1 Macro: {best_metrics['f1_macro']:.4f}")
    print(f"Best F1 Micro: {best_metrics['f1_micro']:.4f}")
    print(f"Best F1 Weighted: {best_metrics['f1_weighted']:.4f}")
    print(f"Best AUC: {best_metrics['auc']:.4f}")

In [ ]:
train_model(
            model, train_loader, val_loader, optimizer,\
            criterion, num_classes=num_classes, epochs=70
            )

False
Epoch 1
Loss: 0.1977
F1 Macro: 0.4825
F1 Weighted: 0.8998
AUC: 0.7676
----------------------------------------
False
Epoch 2
Loss: 0.1929
F1 Macro: 0.4825
F1 Weighted: 0.8998
AUC: 0.7716
----------------------------------------
False
Epoch 3
Loss: 0.1930
F1 Macro: 0.4825
F1 Weighted: 0.8998
AUC: 0.7742
----------------------------------------
False
Epoch 4
Loss: 0.1879
F1 Macro: 0.5007
F1 Weighted: 0.9022
AUC: 0.7745
----------------------------------------
False
Epoch 5
Loss: 0.1799
F1 Macro: 0.5345
F1 Weighted: 0.9073
AUC: 0.7765
----------------------------------------
False
Epoch 6
Loss: 0.1757
F1 Macro: 0.5358
F1 Weighted: 0.9080
AUC: 0.7817
----------------------------------------
False
Epoch 7
Loss: 0.1729
F1 Macro: 0.6253
F1 Weighted: 0.9229
AUC: 0.7830
----------------------------------------
False
Epoch 8
Loss: 0.1678
F1 Macro: 0.6336
F1 Weighted: 0.9234
AUC: 0.7888
----------------------------------------
False
Epoch 9
Loss: 0.1656
F1 Macro: 0.6437
F1 Weighted: 0.9247
